In [1]:
import pandas as pd
import numpy as np
import re
import string
import nltk
from nltk import sent_tokenize
import emoji
import tqdm
from tqdm.auto import tqdm
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.ensemble import RandomForestClassifier

In [2]:
df = pd.read_csv('datasets/IMDB Dataset.csv')

In [3]:
df.head(3)

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive


In [4]:
# check class balance
df['sentiment'].value_counts()

sentiment
positive    25000
negative    25000
Name: count, dtype: int64

In [5]:
# check missing values
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [6]:
# check duplicates
df.duplicated().sum()

np.int64(418)

In [7]:
# drop duplicates
df.drop_duplicates(inplace=True)

In [8]:
# check duplicates again
df.duplicated().sum()

np.int64(0)

In [9]:
# convert text to lowercase
df['review'] = df['review'].str.lower()

In [10]:
# remove html tags
def remove_html(text):
    pattern = re.compile(r'<.*?>')
    return pattern.sub('', text)

In [11]:
df['review'] = df['review'].apply(remove_html)

In [12]:
df.head(3)

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production. the filming tec...,positive
2,i thought this was a wonderful way to spend ti...,positive


In [13]:
# remove URLs
def remove_url(text):
    pattern = re.compile(r'http\S+|www\.\S+')
    return pattern.sub('', text)

In [14]:
df['review'] = df['review'].apply(remove_url)

In [15]:
df.head(3)

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production. the filming tec...,positive
2,i thought this was a wonderful way to spend ti...,positive


In [16]:
# remove multiple spaces
def remove_multiple_spaces(text):
    pattern = re.compile(r'\s+')
    return pattern.sub(' ', text)

In [17]:
df['review'] = df['review'].apply(remove_multiple_spaces)

In [18]:
df.head(3)

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production. the filming tec...,positive
2,i thought this was a wonderful way to spend ti...,positive


In [19]:
# remove punctuations
def remove_punctuations(text):
    return text.translate(str.maketrans('','',string.punctuation))

In [20]:
df['review'] = df['review'].apply(remove_punctuations)

In [21]:
df.head(3)

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production the filming tech...,positive
2,i thought this was a wonderful way to spend ti...,positive


In [22]:
# handle emojis
tqdm.pandas(desc="Processing Data")
df['review'] = df['review'].progress_apply(emoji.demojize)

Processing Data:   0%|          | 0/49582 [00:00<?, ?it/s]

In [23]:
# remove stop words
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Grv\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [24]:
from nltk.corpus import stopwords
stopwords = stopwords.words('english')
def remove_stopwords(text):
    filtered_words = [word for word in text.split() if word not in stopwords]
    return " ".join(filtered_words)

In [25]:
df['review'] = df['review'].progress_apply(remove_stopwords)

Processing Data:   0%|          | 0/49582 [00:00<?, ?it/s]

In [26]:
df.head(3)

,review,sentiment
0,one reviewers mentioned watching 1 oz episode ...,positive
1,wonderful little production filming technique ...,positive
2,thought wonderful way spend time hot summer we...,positive


In [27]:
" ".join(stopwords)

"a about above after again against ain all am an and any are aren aren't as at be because been before being below between both but by can couldn couldn't d did didn didn't do does doesn doesn't doing don don't down during each few for from further had hadn hadn't has hasn hasn't have haven haven't having he he'd he'll her here hers herself he's him himself his how i i'd if i'll i'm in into is isn isn't it it'd it'll it's its itself i've just ll m ma me mightn mightn't more most mustn mustn't my myself needn needn't no nor not now o of off on once only or other our ours ourselves out over own re s same shan shan't she she'd she'll she's should shouldn shouldn't should've so some such t than that that'll the their theirs them themselves then there these they they'd they'll they're they've this those through to too under until up ve very was wasn wasn't we we'd we'll we're were weren weren't we've what when where which while who whom why will with won won't wouldn wouldn't y you you'd you

In [28]:
# X,y split
X = df.iloc[:,:-1]
y = df.iloc[:,-1]

In [29]:
y.head(3)

0    positive
1    positive
2    positive
Name: sentiment, dtype: object

In [30]:
# label encode y
le = LabelEncoder()
y = le.fit_transform(y)

In [31]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=1)

In [32]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((39665, 1), (9917, 1), (39665,), (9917,))

In [33]:
# BOW
cv = CountVectorizer(ngram_range=(1,1), min_df=5, max_df=0.95, max_features=5000)
X_train_bow = cv.fit_transform(X_train['review']).toarray()
X_test_bow = cv.transform(X_test['review']).toarray()

In [34]:
# TFIDF
tfidf = TfidfVectorizer(ngram_range=(1,1), min_df=5, max_df=0.95, max_features=5000)
X_train_tfidf = tfidf.fit_transform(X_train['review']).toarray()
X_test_tfidf = tfidf.transform(X_test['review']).toarray()

In [35]:
# Gaussian Naive Bayes Model with BOW
gnb = GaussianNB()
gnb.fit(X_train_bow, y_train)
y_pred1 = gnb.predict(X_test_bow)

In [36]:
confusion_matrix(y_test, y_pred1), accuracy_score(y_test, y_pred1), f1_score(y_test, y_pred1)

(array([[4318,  715],
        [1576, 3308]]),
 0.7689825552082283,
 0.742786572358819)

In [37]:
# Gaussian Naive Bayes Model with TFIDF
gnb = GaussianNB()
gnb.fit(X_train_tfidf, y_train)
y_pred12 = gnb.predict(X_test_tfidf)

In [38]:
confusion_matrix(y_test, y_pred12), accuracy_score(y_test, y_pred12), f1_score(y_test, y_pred12)

(array([[4102,  931],
        [ 937, 3947]]),
 0.8116365836442473,
 0.8086457693095677)

In [39]:
# Random Forest Model with BOW
rf = RandomForestClassifier(n_estimators=300, max_depth=5, n_jobs=-1, verbose=1, random_state=42)
rf.fit(X_train_bow, y_train)
y_pred2 = rf.predict(X_test_bow)

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 20 concurrent workers.
[Parallel(n_jobs=-1)]: Done  10 tasks      | elapsed:    0.8s
[Parallel(n_jobs=-1)]: Done 160 tasks      | elapsed:    6.7s
[Parallel(n_jobs=-1)]: Done 300 out of 300 | elapsed:   12.1s finished
[Parallel(n_jobs=20)]: Using backend ThreadingBackend with 20 concurrent workers.
[Parallel(n_jobs=20)]: Done  10 tasks      | elapsed:    0.0s
[Parallel(n_jobs=20)]: Done 160 tasks      | elapsed:    0.0s
[Parallel(n_jobs=20)]: Done 300 out of 300 | elapsed:    0.1s finished


In [40]:
confusion_matrix(y_test, y_pred2), accuracy_score(y_test, y_pred2), f1_score(y_test, y_pred2)

(array([[3560, 1473],
        [ 452, 4432]]),
 0.8058888776847837,
 0.821577532672166)

In [41]:
# Random Forest Model with TFIDF
rf = RandomForestClassifier(n_estimators=300, max_depth=5, n_jobs=-1, verbose=1, random_state=42)
rf.fit(X_train_tfidf, y_train)
y_pred22 = rf.predict(X_test_tfidf)

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 20 concurrent workers.
[Parallel(n_jobs=-1)]: Done  10 tasks      | elapsed:    0.8s
[Parallel(n_jobs=-1)]: Done 160 tasks      | elapsed:    7.4s
[Parallel(n_jobs=-1)]: Done 300 out of 300 | elapsed:   13.5s finished
[Parallel(n_jobs=20)]: Using backend ThreadingBackend with 20 concurrent workers.
[Parallel(n_jobs=20)]: Done  10 tasks      | elapsed:    0.0s
[Parallel(n_jobs=20)]: Done 160 tasks      | elapsed:    0.0s
[Parallel(n_jobs=20)]: Done 300 out of 300 | elapsed:    0.1s finished


In [42]:
confusion_matrix(y_test, y_pred22), accuracy_score(y_test, y_pred22), f1_score(y_test, y_pred22)

(array([[3593, 1440],
        [ 453, 4431]]),
 0.8091156599778159,
 0.8239888423988843)

# Word2Vec

In [43]:
nltk.download('punkt_tab')
story = []
for doc in df['review']:
    sentences = sent_tokenize(doc)
    for sentence in sentences:
        story.append(simple_preprocess(sentence))

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Grv\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [44]:
# Word2Vec - method 1
# model_wv = Word2Vec(window=10, vector_size=300, min_count=2, workers=12)
# model_wv.build_vocab(story)
# model_wv.train(story, total_examples=model_wv.corpus_count, epochs=model_wv.epochs)

In [45]:
# Word2Vec - method 2
model_wv = Word2Vec(sentences=story, window=10, vector_size=300, min_count=2, workers=8)

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


In [46]:
len(model_wv.wv.index_to_key)

79940

In [47]:
# get document embeddings from word embeddings
def doc_vector(doc):
    doc = [word for word in doc.split() if word in model_wv.wv.index_to_key]
    return np.mean(model_wv.wv[doc], axis=0)

In [48]:
# apply above method
X = []
for doc in tqdm(df['review'].values):
    X.append(doc_vector(doc))

  0%|          | 0/49582 [00:00<?, ?it/s]

In [49]:
len(X), len(X[0]), len(X[1]), len(X[2])

(49582, 300, 300, 300)

In [50]:
model_wv.wv.get_normed_vectors()

array([[ 0.02346182, -0.04029215,  0.00553534, ...,  0.03298963,
         0.00857695, -0.05195884],
       [-0.02162589,  0.03836122, -0.04770984, ..., -0.04663769,
         0.03734647, -0.06810834],
       [ 0.09195084, -0.04680834, -0.00471769, ..., -0.1108299 ,
         0.05984654,  0.00376359],
       ...,
       [-0.01038514,  0.16040896,  0.00597456, ..., -0.0282784 ,
        -0.00233341,  0.01095178],
       [-0.01990778,  0.13928893,  0.03418959, ..., -0.06859661,
         0.02453507, -0.05776246],
       [-0.03626021,  0.0390103 ,  0.01482256, ..., -0.0494568 ,
         0.01215481, -0.08475514]], shape=(79940, 300), dtype=float32)

In [51]:
model_wv.wv.index_to_key

['movie',
 'film',
 'one',
 'like',
 'good',
 'even',
 'would',
 'time',
 'really',
 'see',
 'story',
 'much',
 'well',
 'get',
 'great',
 'bad',
 'also',
 'people',
 'first',
 'dont',
 'movies',
 'made',
 'films',
 'make',
 'could',
 'way',
 'characters',
 'think',
 'watch',
 'many',
 'seen',
 'character',
 'two',
 'never',
 'love',
 'acting',
 'best',
 'plot',
 'little',
 'know',
 'show',
 'life',
 'ever',
 'better',
 'say',
 'still',
 'scene',
 'end',
 'man',
 'scenes',
 'something',
 'go',
 'back',
 'real',
 'im',
 'watching',
 'thing',
 'doesnt',
 'didnt',
 'actors',
 'years',
 'actually',
 'though',
 'makes',
 'funny',
 'another',
 'find',
 'nothing',
 'look',
 'going',
 'work',
 'lot',
 'new',
 'every',
 'old',
 'us',
 'part',
 'cant',
 'director',
 'want',
 'quite',
 'thats',
 'things',
 'pretty',
 'cast',
 'seems',
 'around',
 'young',
 'got',
 'take',
 'fact',
 'world',
 'enough',
 'big',
 'horror',
 'thought',
 'give',
 'may',
 'ive',
 'however',
 'without',
 'long',
 'saw',

In [52]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=1)

In [53]:
rf = RandomForestClassifier(n_estimators=100, max_depth=8, n_jobs=-1, verbose=1)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 20 concurrent workers.
[Parallel(n_jobs=-1)]: Done  10 tasks      | elapsed:    2.7s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:   14.1s finished
[Parallel(n_jobs=20)]: Using backend ThreadingBackend with 20 concurrent workers.
[Parallel(n_jobs=20)]: Done  10 tasks      | elapsed:    0.0s
[Parallel(n_jobs=20)]: Done 100 out of 100 | elapsed:    0.0s finished


In [54]:
confusion_matrix(y_test, y_pred), accuracy_score(y_test, y_pred), f1_score(y_test, y_pred)

(array([[4108,  925],
        [ 699, 4185]]),
 0.8362407986286176,
 0.8375025015009006)